# 🧠 Fine-Tuning & Evaluation Pipeline für Fraud Detection

## 📋 Projekt-Übersicht & Voraussetzungen

Dieses Notebook implementiert eine vollständige End-to-End-Pipeline für das Supervised Fine-Tuning (SFT) und die anschließende Evaluierung eines Sprachmodells (**Qwen/Qwen2-1.5B-Instruct**) zur Erkennung von betrügerischen Telefonaten (Fraud Detection).

### ⚙️ Systemvoraussetzungen (Requirements)
* **Docker** mit dem NVIDIA Container Toolkit.
* Ein **NVIDIA NGC API Key** (erforderlich für Registrierung/Zugriff).
* Eine **NVIDIA GPU** mit mindestens 20 GB VRAM.
* *Hinweis zu Llama-Modellen:* Falls du stattdessen Llama-Modelle nutzen möchtest, musst du vorher den entsprechenden Zugriff auf Hugging Face anfordern.

### 📊 Geschätzter VRAM-Bedarf nach Konfiguration
| Konfiguration | Modell | Methode | Geschätzter VRAM |
| :--- | :--- | :--- | :--- |
| `qwen2_1p5b_sft.yaml` | Qwen2-1.5B-Instruct | Full SFT | ~20 GB |
| `llama3p2_3b_sft.yaml` | Llama-3.2-3B-Instruct | Full SFT | ~38 GB |
| `llama3p1_8b_lora.yaml` | Llama-3.1-8B | LoRA | ~24 GB |

💡 **Hinweis zu den Konfigurationsdateien:** Unter dem Verzeichnis `Automodel_Finetuning_Configs` findest du die entsprechenden YAML-Configdateien, die du bei Bedarf für die verschiedenen Modelle anpassen kannst.

---

⚠️ **Wichtiger Hinweis zur Umgebung:** Dieses Notebook muss zwingend im **Automodel Container** ausgeführt werden, da dort alle benötigten Bibliotheken (wie PyTorch, Transformers, PEFT und das `automodel`-CLI-Tool) vorinstalliert sind.

### Die Pipeline gliedert sich in folgende Hauptschritte:
1. **Environment & GPU Check:** Überprüfung von NVIDIA-Treiber (`nvidia-smi`), CUDA-Verfügbarkeit in PyTorch sowie der Existenz des `automodel`-CLI-Tools.
2. **Data Preparation:** Formatierung der rohen JSONL-Trainings- und Testdaten in ein strukturiertes Prompt-Antwort-Schema.
3. **Training:** Ausführung des Fine-Tunings über `automodel` (wahlweise als schneller Smoke-Test mit 50 Schritten oder als vollständiges Training).
4. **Evaluation & Metriken:** Generierung von Vorhersagen für das Basismodell sowie das feingetunte Modell auf den Testdaten, gefolgt von einer automatisierten Klassifikationsauswertung (`classification_report` und Confusion Matrix).

In [1]:
# Import aller benötigten Bibliotheken
import json
import shutil
import subprocess
import time
from pathlib import Path
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from sklearn.metrics import classification_report, confusion_matrix

print("✅ Bibliotheken erfolgreich importiert.")

/usr/local/lib/python3.10/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ Bibliotheken erfolgreich importiert.


### 1. Environment & GPU Check
Hier wird sichergestellt, dass die GPU ordnungsgemäß funktioniert und das `automodel`-Werkzeug im Container bereitsteht.

In [2]:
def check_environment():
    print("=== Überprüfe Umgebung und GPU (Automodel Container) ===")
    
    # nvidia-smi Check
    try:
        smi_output = subprocess.run(
            ["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv"],
            capture_output=True, text=True, check=True
        ).stdout
        print(smi_output)
    except Exception as e:
        print(f"Warnung bei nvidia-smi: {e}")

    # PyTorch CUDA Check
    print("CUDA verfügbar:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("Device:", torch.cuda.get_device_name(0))
        print("VRAM (GB):", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1))
    assert torch.cuda.is_available(), "Keine GPU sichtbar — stelle sicher, dass der Container mit --gpus all gestartet wurde."

    # automodel CLI Check
    automodel_path = shutil.which("automodel")
    assert automodel_path, "automodel CLI nicht gefunden — bitte stelle sicher, dass du dich im offiziellen Automodel Container befindest!"
    print("automodel CLI gefunden unter:", automodel_path)

check_environment()

=== Überprüfe Umgebung und GPU (Automodel Container) ===
name, memory.total [MiB], driver_version
NVIDIA L40S, 46068 MiB, 580.126.09

CUDA verfügbar: True
Device: NVIDIA L40S
VRAM (GB): 47.7


AssertionError: automodel CLI nicht gefunden — bitte stelle sicher, dass du dich im offiziellen Automodel Container befindest!

### 2. Data Preparation
Die Transkripte werden mittels eines einheitlichen Prompt-Templates in das Trainingsformat für das Sprachmodell überführt.

In [ ]:
PROMPT_TEMPLATE = (
    "Phone call transcript: {text}\n\n"
    "Question: Based on the phone call transcript above, is it a fraudulent call or a legitimate call?\n\n"
    "Answer: "
)

def prepare_data():
    print("\n=== Bereite Daten vor ===")
    def to_record(entry):
        return {
            "input": PROMPT_TEMPLATE.format(text=entry["input"]),
            "output": entry["output"]
        }

    data_files = ["data/raw/train.jsonl", "data/raw/test.jsonl", "data/raw/validation.jsonl"]
    for data_path in data_files:
        path_obj = Path(data_path)
        if not path_obj.exists():
            print(f"Überspringe {data_path}, da nicht vorhanden.")
            continue
            
        with open(path_obj) as raw_data_file:
            records = [to_record(json.loads(line)) for line in raw_data_file]
            processed_data_path = Path("data") / path_obj.name
            processed_data_path.parent.mkdir(parents=True, exist_ok=True)
            with open(processed_data_path, "w") as processed_data_file:
                for record in records:
                    processed_data_file.write(json.dumps(record) + "\n")
        print(f"Verarbeitet und gespeichert: {processed_data_path}")

prepare_data()

### 3. Training
Ausführung des Fine-Tunings über das `automodel`-Framework. Du kannst hier wählen, ob ein schneller Smoke-Test (z. B. 50 Schritte) oder das vollständige Training ausgeführt werden soll.

In [ ]:
def run_training(smoke_test=False):
    print("\n=== Starte Training ===")
    config_path = "configs/qwen2_1p5b_sft.yaml"
    
    if smoke_test:
        print("Führe Smoke-Test aus (50 Schritte)...")
        subprocess.run([
            "automodel", config_path,
            "--step_scheduler.max_steps", "50",
            "--checkpoint.checkpoint_dir", "results/qwen2_1p5b_sft_smoke_test/checkpoints"
        ], check=True)
    else:
        print("Führe vollständiges Training aus...")
        subprocess.run(["automodel", config_path], check=True)

# Wähle hier aus (True für Smoke-Test, False für Volltraining):
run_training(smoke_test=True)

### 4. Evaluation Helper Functions & Runner
Diese Sektion lädt das Modell (inklusive möglicher PEFT/LoRA-Adapter), generiert stapelweise Vorhersagen (`generate_batch`) und vergleicht das Basismodell im Zero-Shot-Vergleich mit dem feingetunten Modell anhand von Metriken wie Precision, Recall und der Confusion Matrix.

In [ ]:
def load_model(checkpoint: str):
    is_local_peft_adapter = Path(checkpoint).is_dir() and (Path(checkpoint) / "adapter_config.json").exists()
    if is_local_peft_adapter:
        from peft import AutoPeftModelForCausalLM
        print(f"Lade PEFT/LoRA Adapter von {checkpoint}")
        model = AutoPeftModelForCausalLM.from_pretrained(checkpoint, torch_dtype=torch.bfloat16, device_map="cuda")
        tokenizer_source = checkpoint
    else:
        print(f"Lade Basismodell {checkpoint}")
        model = AutoModelForCausalLM.from_pretrained(checkpoint, torch_dtype=torch.bfloat16, device_map="cuda")
        tokenizer_source = checkpoint

    tokenizer = AutoTokenizer.from_pretrained(tokenizer_source)
    if tokenizer.pad_token_id is None:
        tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "left"
    model.eval()
    return model, tokenizer

def generate_batch(model, tokenizer, prompts, max_new_tokens=16):
    inputs = tokenizer(prompts, return_tensors="pt", padding=True, truncation=True, max_length=2048).to(model.device)
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
        )
    completions = output_ids[:, inputs["input_ids"].shape[1]:]
    return tokenizer.batch_decode(completions, skip_special_tokens=True, clean_up_tokenization_spaces=False)

def generate_predictions(checkpoint: str, input_jsonl: str, output_jsonl: str, limit: int = None, max_batch_size: int = 8):
    with open(input_jsonl) as f:
        records = [json.loads(line) for line in f]
    if limit:
        records = records[:limit]

    model, tokenizer = load_model(checkpoint)
    predictions = []
    batch_timings = []
    
    for i in range(0, len(records), max_batch_size):
        batch = records[i : i + max_batch_size]
        prompts = [r["input"] for r in batch]
        start = time.perf_counter()
        predictions.extend(generate_batch(model, tokenizer, prompts))
        batch_timings.append({"n": len(batch), "seconds": time.perf_counter() - start})

    out_path = Path(output_jsonl)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    with open(out_path, "w") as f:
        for record, prediction in zip(records, predictions):
            out = dict(record)
            out["prediction"] = prediction.strip()
            f.write(json.dumps(out) + "\n")

    print(f"Vorhersagen geschrieben nach {output_jsonl}")

def evaluate_models():
    print("\n=== Starte Evaluierung ===")
    BASE_MODEL = "Qwen/Qwen2-1.5B-Instruct"
    RESULTS_DIR = Path("results")
    RUN_NAME = "qwen2-1p5b-sft"

    checkpoint_path = RESULTS_DIR / RUN_NAME / "checkpoints" / "LATEST" / "model"
    if (checkpoint_path / "consolidated").exists():
        checkpoint_path = checkpoint_path / "consolidated"
        
    assert checkpoint_path.exists(), f"Checkpoint {checkpoint_path} nicht gefunden — bitte erst das Training ausführen."

    EVAL_N = 200
    base_output = str(RESULTS_DIR / "eval_base_predictions.jsonl")
    finetuned_output = str(RESULTS_DIR / "eval_finetuned_predictions.jsonl")
    test_jsonl = str(Path("data") / "test.jsonl")

    # Generierung starten
    generate_predictions(BASE_MODEL, test_jsonl, base_output, EVAL_N)
    generate_predictions(str(checkpoint_path), test_jsonl, finetuned_output, EVAL_N)

    def normalize(text):
        text = str(text).strip().lower()
        if "fraud" in text:
            return "fraud"
        if "legitimate" in text:
            return "legitimate"
        return "unparsed"

    def load_preds(output_path):
        with open(output_path) as f:
            rows = [json.loads(line) for line in f]
        return [normalize(r["output"]) for r in rows], [normalize(r["prediction"]) for r in rows]

    true_labels, base_preds = load_preds(base_output)
    _, finetuned_preds = load_preds(finetuned_output)

    results = [("Base model (zero-shot)", base_preds), ("Fine-tuned", finetuned_preds)]

    for label, preds in results:
        print(f"\n=== {label} ===")
        print(classification_report(true_labels, preds, labels=["fraud", "legitimate"], zero_division=0))
        print("Confusion matrix (rows=true, cols=predicted, order=[fraud, legitimate]):")
        print(confusion_matrix(true_labels, preds, labels=["fraud", "legitimate"]))
        unparsed = sum(1 for p in preds if p == "unparsed")
        if unparsed:
            print(f"({unparsed} Vorhersagen konnten nicht als fraud/legitimate geparst werden)")

# Evaluierung ausführen
evaluate_models()